In [8]:
import os
import random
from collections import defaultdict

import plotly.express as px
from pandas import DataFrame
from torch import load, stack

from utils import centroid

In [9]:
datasets = {
    'core_clinical': {
        'folder': 'core_clinical_short_gemma3_27b_21/',
        'files': 'core_clinical_short'
    },
    'books': {
        'folder': 'books_gemma3_27b_21/',
        'files': 'books'
    },
    'positive_affect': {
        'folder': 'positive_affect_gemma3_27b_21/',
        'files': 'positive_affect'
    },
    'happydb': {
        'folder': 'happydb_gemma3_27b_21/',
        'files': 'happydb'
    },
    'redsm5': {
        'folder': 'redsm5_gemma3_27b_21/',
        'files': 'redsm5'
    }
}

In [10]:
# Contrastive depression DIRECTION, scored by cosine similarity. This is
# not an orthogonalized basis axis, and not the oblique projection used in 75 --
# it is a single contrast between two corpus centroids, scored by cosine so the
# result depends on direction only and is bounded in [-1, +1].
#
#   direction = centroid(core_clinical) - centroid(positive_affect)
#   score(x)  = cos_sim( centroid(x), direction )
#             = (c . d) / (||c|| ||d||)   -- d pre-normalized once, outside the loop

def corpus_centroid(folder, text_dir):
    """Token-weighted mean embedding over every text in a corpus."""
    total, n = None, 0
    for text in os.listdir(text_dir):
        token_emb = load(folder + text + '_tensor')[0][1:].double()  # (n_tok, D), drop BOS
        s = token_emb.sum(dim=0)
        total = s if total is None else total + s
        n += token_emb.shape[0]
    return total / n


def symptom_centroids(folder):
    """Token-weighted mean layer-21 embedding per symptom, computed directly from
    the core_clinical embeddings. Symptom label is the part after the comma in each
    filename, e.g. 'dsm-5,mood_tensor' -> 'mood'."""
    totals, counts = {}, {}
    for fname in os.listdir(folder):
        if not fname.endswith('_tensor'):
            continue
        symptom = fname[:-len('_tensor')].split(',')[1]
        token_emb = load(folder + fname)[0][1:].double()  # (n_tok, D), drop BOS
        s = token_emb.sum(dim=0)
        totals[symptom] = s if symptom not in totals else totals[symptom] + s
        counts[symptom] = counts.get(symptom, 0) + token_emb.shape[0]
    return {k: totals[k] / counts[k] for k in totals}


symptom_centroid = symptom_centroids(datasets['core_clinical']['folder'])
core_centroid = centroid(stack(list(symptom_centroid.values())))          # depression pole
positive_centroid = corpus_centroid(datasets['positive_affect']['folder'],  # positive-affect pole
                                    datasets['positive_affect']['files'])

direction = core_centroid - positive_centroid   # depression <- -> positive-affect
unit_direction = direction / direction.norm()


def depression_cosine(c):
    """Cosine similarity between a text centroid and the depression direction, in [-1, 1]."""
    return (c @ unit_direction) / c.norm()


# Sanity check: the clinical pole should sit above the positive pole.
# No mean is subtracted anywhere, so every score keeps the shared common mode and
# stays in a narrow POSITIVE band (~0.18-0.25) whatever the corpus -- read the
# ORDERING, not the absolute values, and note the sign is not informative.
print({'core_clinical pole': depression_cosine(core_centroid).item(),
       'positive_affect pole': depression_cosine(positive_centroid).item()})

{'core_clinical pole': 0.23191479528652742, 'positive_affect pole': 0.19671739219790002}


In [11]:
# Score every text: cosine similarity to the depression direction, in [-1, +1].
#
# Sampling keeps the two held-out corpora balanced: books are used in full and
# redsm5 tops each symptom up to the largest books symptom (141), giving 423
# naturalistic texts; happydb is truncated to the same 423.
SEED = 0

_books_n = defaultdict(int)
for _f in os.listdir(datasets["books"]["files"]):
    _books_n[_f.split(",")[1]] += 1
_target = max(_books_n.values())
_happy_n = 3 * _target


def select_files(text_dir, dataset_name):
    """Filenames to include for a dataset (deterministic given SEED)."""
    files = sorted(os.listdir(text_dir))

    if dataset_name == "redsm5":
        by_symptom = defaultdict(list)
        for f in files:
            by_symptom[f.split(",")[1]].append(f)
        rng = random.Random(SEED)
        selected = []
        for symptom in sorted(by_symptom):
            need = max(0, _target - _books_n[symptom])
            selected += rng.sample(by_symptom[symptom], min(need, len(by_symptom[symptom])))
        return sorted(selected)

    if dataset_name == "happydb":
        return files[:_happy_n]

    return files


rows = []
for dataset_name, dataset_info in datasets.items():
    folder = dataset_info['folder']
    text_dir = dataset_info['files']

    for text in select_files(text_dir, dataset_name):
        token_emb = load(folder + text + '_tensor')[0][1:]  # per-token embeddings, drop BOS
        centroid_rs = centroid(token_emb.double())          # float64 BEFORE averaging

        with open(os.path.join(text_dir, text), encoding="utf-8") as f:
            plain_text = f.read()

        rows.append([text,
                     depression_cosine(centroid_rs).item(),
                     centroid_rs.norm().item(),  # magnitude the cosine divides out
                     text.split(",")[1],
                     dataset_name,
                     plain_text,
                     token_emb.shape[0]])

projection_gram = DataFrame(rows,
                            columns=["filename", "depression", "centroid_norm",
                                     "clinician_annotation", "dataset",
                                     "plain_text", "n_tokens"])

# Visualization-only relabelling: books + redsm5 fold into one "naturalistic" corpus
# (as in 75) and happydb becomes the happy control. Keep in sync with CORPUS_ORDER
# below -- the figure asserts on the mismatch.
DISPLAY_DATASET = {"books": "naturalistic", "redsm5": "naturalistic",
                   "happydb": "control_happy"}
projection_gram["dataset"] = projection_gram["dataset"].replace(DISPLAY_DATASET)

print(projection_gram.groupby(["dataset", "clinician_annotation"], observed=True).size())

dataset          clinician_annotation
control_happy    control                 423
core_clinical    mood                     19
                 somatic                  24
                 suicidality               8
naturalistic     mood                    141
                 somatic                 141
                 suicidality             141
positive_affect  positive                  9
dtype: int64


In [12]:
# Affect discrimination. naturalistic vs control_happy is the key test: both are
# held out (neither defines the direction) and both are n=423. The pole comparison is
# in-sample, so it is an upper bound, not evidence of generalization. The magnitude
# check must come out near chance or below -- otherwise the score is partly reading
# passage length/register rather than affect (the cosine divides magnitude out).
#
# AUC = P(a random passage from group a scores above one from group b) = U/(n1*n2);
# 0.5 = chance, 1.0 = perfect. p is one-sided (a > b) and UNCORRECTED -- these
# are two pre-specified comparisons, not the 9-test BH-adjusted family in 75.
# Figures show stars only; the exact p values printed here are for the caption.
from scipy.stats import mannwhitneyu

SCORE = "depression"

HELD_OUT = ("naturalistic", "control_happy")
IN_SAMPLE = ("core_clinical", "positive_affect")

_by = {d: g for d, g in projection_gram.groupby("dataset", observed=True)}


def auc(a, b, score=SCORE):
    """Mann-Whitney AUC and one-sided p for corpus `a` > corpus `b` on `score`."""
    x, y = _by[a][score].to_numpy(), _by[b][score].to_numpy()
    res = mannwhitneyu(x, y, alternative="greater")
    return res.statistic / (x.size * y.size), res.pvalue


def stars(p):
    """Significance stars, same thresholds as the Dunn brackets in 75."""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


AUC_HELD, P_HELD = auc(*HELD_OUT)
AUC_POLE, P_POLE = auc(*IN_SAMPLE)
auc_norm, _ = auc(*HELD_OUT, score="centroid_norm")

for label, (a, b), (u, p) in [("held-out ", HELD_OUT, (AUC_HELD, P_HELD)),
                              ("in-sample", IN_SAMPLE, (AUC_POLE, P_POLE))]:
    print(f"{label}  {a} (n={len(_by[a])}) > {b} (n={len(_by[b])}):  "
          f"AUC={u:.3f}  p={p:.2e} {stars(p)}   "
          f"medians {_by[a][SCORE].median():.4f} vs {_by[b][SCORE].median():.4f}")

print(f"nuisance   centroid magnitude alone: AUC={auc_norm:.3f} "
      f"({'below chance' if auc_norm < 0.5 else 'PREDICTIVE -- investigate'})")

held-out   naturalistic (n=423) > control_happy (n=423):  AUC=0.789  p=2.51e-48 ***   medians 0.2285 vs 0.2180
in-sample  core_clinical (n=51) > positive_affect (n=9):  AUC=1.000  p=1.06e-06 ***   medians 0.2307 vs 0.1968
nuisance   centroid magnitude alone: AUC=0.273 (below chance)


In [13]:
# Per-symptom companion table: one contrastive direction per symptom, scored the
# same way. These directions are NOT orthogonal to one another -- each is an
# independent contrast against the same positive-affect pole.
#
#   direction_s = centroid(symptom s) - centroid(positive_affect)
#   coord_s(x)  = cos_sim(centroid(x), direction_s)             # in [-1, +1]
SYMPTOMS_3D = ["mood", "somatic", "suicidality"]

sym_direction = {s: symptom_centroid[s] - positive_centroid for s in SYMPTOMS_3D}
sym_unit = {s: sym_direction[s] / sym_direction[s].norm() for s in SYMPTOMS_3D}


def symptom_cosine(c, s):
    """Cosine similarity between a text centroid and symptom s's direction, in [-1, 1]."""
    return (c @ sym_unit[s]) / c.norm()


rows_3d = []
for dataset_name, dataset_info in datasets.items():
    folder = dataset_info['folder']
    text_dir = dataset_info['files']

    for text in select_files(text_dir, dataset_name):  # same subsample as projection_gram
        token_emb = load(folder + text + '_tensor')[0][1:]  # per-token embeddings (drop BOS)
        centroid_rs = centroid(token_emb.double())          # float64 BEFORE averaging

        with open(os.path.join(text_dir, text), encoding="utf-8") as f:
            plain_text = f.read()

        rows_3d.append([text,
                        symptom_cosine(centroid_rs, "mood").item(),
                        symptom_cosine(centroid_rs, "somatic").item(),
                        symptom_cosine(centroid_rs, "suicidality").item(),
                        text.split(",")[1],
                        dataset_name,
                        plain_text,
                        token_emb.shape[0]])  # true token length the centroid averaged over

projection_3d_df = DataFrame(rows_3d, columns=["filename", "mood", "somatic", "suicidality",
                                               "clinician_annotation", "dataset",
                                               "plain_text", "n_tokens"])

# Same visualization-only relabelling as the depression-score table above.
projection_3d_df["dataset"] = projection_3d_df["dataset"].replace(DISPLAY_DATASET)

In [14]:
# Cosine similarity to the depression direction, by corpus, with the two AUC
# comparisons printed above as brackets. The middle two corpora are held out
# (neither defines the direction); the outer two define it, so their separation
# is an in-sample upper bound.
# Higher = nearer the depression pole; no zero line because an uncentred cosine
# never crosses zero here (all scores ~0.18-0.25).
import textwrap

# Two Okabe-Ito hues, one per affect: blue = depressive, pink = positive. The
# in-sample/held-out split is carried by the bracket labels, not by colour.
DEPRESSIVE, POSITIVE = "#0072B2", "#CC79A7"

# Page geometry (pixels; kaleido writes 0.75 pt per px). Matched to the
# figure-3 panels so both print at about the same scale.
FIG_W, FIG_H = 560, 480
CORPUS_COLOR = {
    "core_clinical": DEPRESSIVE,
    "naturalistic": DEPRESSIVE,
    "control_happy": POSITIVE,
    "positive_affect": POSITIVE,
}
CORPUS_ORDER = ["core_clinical", "naturalistic", "control_happy", "positive_affect"]
# Formal paper labels; dict keys stay snake_case to match the data.
CORPUS_LABEL = {
    "core_clinical": "Core Clinical<br>(Depressed)",
    "naturalistic": "Naturalistic<br>(Depressed)",
    "control_happy": "HappyDB",
    "positive_affect": "Positive Affect",
}
assert not set(_by) - set(CORPUS_ORDER), f"corpora would be dropped: {set(_by) - set(CORPUS_ORDER)}"

plot_df = projection_gram.copy()
# Wrapped passage text for the hover tooltip; N folded into the x tick labels.
plot_df["hover_text"] = plot_df["plain_text"].map(
    lambda t: "<br>".join(textwrap.wrap(t, width=70)))
_tick = {d: f"{CORPUS_LABEL[d]}<br>n={len(_by[d])}" for d in CORPUS_ORDER}
plot_df["corpus_label"] = plot_df["dataset"].map(_tick)

fig = px.box(plot_df, y=SCORE, x="corpus_label", color="dataset", points="all",
             hover_data=["filename", "clinician_annotation", "hover_text"],
             category_orders={"corpus_label": [_tick[d] for d in CORPUS_ORDER],
                              "dataset": CORPUS_ORDER},
             color_discrete_map=CORPUS_COLOR)

fig.update_layout(
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=13, color="black"),
    showlegend=False,  # x tick labels already name every corpus
    boxmode="overlay",  # x == color, so keep full-width boxes
    boxgap=0.5,
    margin=dict(l=80, r=24, t=24, b=70),   # no title to leave room for
    width=FIG_W, height=FIG_H,   # kept in sync with write_image below
)
fig.update_traces(offsetgroup=None, alignmentgroup=None, jitter=0.35, pointpos=0,
                  marker=dict(size=3.5, opacity=0.45, line=dict(width=0)),
                  line=dict(width=1.7))
fig.update_xaxes(title_text="", tickfont=dict(size=12), showgrid=False, linecolor="black",
                 linewidth=0.8, ticks="outside", ticklen=4, tickwidth=0.8, automargin=True)
fig.update_yaxes(title_text="Cosine similarity to depression direction",
                 title_font=dict(size=14),
                 tickfont=dict(size=12), title_standoff=6, automargin=True, showgrid=True,
                 gridcolor="rgba(200,200,200,0.3)", gridwidth=0.6, zeroline=False,
                 linecolor="black", linewidth=0.8, ticks="outside", ticklen=4, tickwidth=0.8)

# Numeric x coordinates index the categories in category_orders, so a bracket spans
# the two indices. Two tiers keep the wide in-sample bracket clear of the narrow one.
_idx = {d: i for i, d in enumerate(CORPUS_ORDER)}
_lo, _hi = plot_df[SCORE].min(), plot_df[SCORE].max()
_span = _hi - _lo
for (a, b), u, p, lift, kind in [(HELD_OUT, AUC_HELD, P_HELD, 0.10, "Naturalistic (Depressed) > HappyDB · Held-Out"),
                                 (IN_SAMPLE, AUC_POLE, P_POLE, 0.32, "Core Clinical (Depressed) > Positive Affect · In-Sample")]:
    y = _hi + _span * lift
    fig.add_shape(type="path", line=dict(color="black", width=1),
                  path=(f"M {_idx[a]},{y - _span * 0.025} L {_idx[a]},{y} "
                        f"L {_idx[b]},{y} L {_idx[b]},{y - _span * 0.025}"))
    fig.add_annotation(x=(_idx[a] + _idx[b]) / 2, y=y, yanchor="bottom", showarrow=False,
                       font=dict(size=12, color="black"), align="center",
                       text=f"{kind}<br>AUC={u:.3f} {stars(p)}")

fig.update_yaxes(range=[_lo - _span * 0.06, _hi + _span * 0.46])

# Vector export for the manuscript (npj: the caption carries the title,
# so the figure itself sets none).
# width/height are REQUIRED here: kaleido ignores layout.width/height and
# would otherwise export at its 700x500 default.
fig.write_image(os.path.join("manuscript", "5.contrastive.pdf"),
                width=FIG_W, height=FIG_H)

fig.show()
